# 2. Data curation: chains and ligands <a id="2"></a>
In this section we will curate our kinase dataset for input into conformational analysis.


## Table of contents

- [2.1 Extracting protein chains](#21)
- [2.2 Extracting small molecules](#22)
- [2.3 Build KLIFS directories](#23)


## Backend map

How this notebook connects to `workflow/` modules (arrows point into the notebook; includes transitive `workflow` subdependencies):

![Backend map](images/backend_maps/02-ChainsAndLigands.v2.svg)

<!-- mermaid source (GitHub does not render mermaid in .ipynb; SVG above is for GitHub):
```mermaid
%%{init: {"flowchart": {"nodeSpacing": 12, "rankSpacing": 28, "padding": 4}, "themeVariables": {"fontSize": "11px"}} }%%
flowchart LR
  NB["02-ChainsAndLigands.ipynb"]
  m_chain_basenames["chain_basenames"]
  m_klifs_filter["klifs_filter"]
  m_pdb_chain_extractor["pdb_chain_extractor"]
  m_utilities["utilities"]
  m_chain_basenames --> m_klifs_filter
  m_chain_basenames --> m_utilities
  m_klifs_filter --> NB
  m_pdb_chain_extractor --> NB
  m_utilities --> NB
```
-->


Our data curation pipeline is subdivided in the following sections: 
2. [Data curation](#2)   
    2.1. [Extracting protein chains](#21)   
    2.2. [Extracting small molecules](#22)   
    2.3. [Build KLIFS directories](#23)


To get started, let's load some packages!

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import shutil
import pandas as pd

from workflow.klifs_filter import KLIFSOverlap
from workflow.pdb_chain_extractor import PDBChainExtractor
from workflow.utilities import PDBDownloader
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs, copy_motif_filtered_datasets


## 2.1 Extracting protein chains <a id="21"></a>
Here we extract only the protein chains containing a kinase domain from our database of downloaded PDB structures.

We utilise the class `PDBChainExtractor()` to write to PDB files the coordinates of chains indicated by InterPro 
query output. 

In [ ]:
from workflow.pdb_chain_extractor import PDBChainExtractor

# Load InterPro table + download flags (written in the download-check cell)
pdb_data_for_extraction_path = "Results/pdb_data_for_chain_extraction.tsv"
pdb_data = pd.read_csv(pdb_data_for_extraction_path, sep="\t", header=0, engine="python")
pdb_data["Accession"] = pdb_data["Accession"].astype(str).str.upper()
if "Downloaded" in pdb_data.columns:
    pdb_data["Downloaded"] = pdb_data["Downloaded"].astype(str).str.lower().isin(("true", "1"))

chain_extractor = PDBChainExtractor()

# --- Dataset 1: protein-only chains (no small molecules) ---
chain_extractor.extract_chains_parallel(
    pdb_data,
    target_dir="Results/InterPro_protein_chains/",
    max_workers=None,
    include_small_molecules=False,
    show_progress=True,
    show_errors=False,
)

Let's make sure that the number of chains corresponds to at least the same amount of files downloaded.

In [ ]:
pdb_directory = 'Results/InterPro_protein_chains/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.2 Extracting small molecules <a id="22"></a>
Here we extract both the kinase protein chains and any small molecule from the PDB.

We utilise the class `PDBChainExtractor()` to write to PDB files of the atomic coordinates of chains and any small 
molecule within 5 Å of the chain. 

In [ ]:
from workflow.pdb_chain_extractor import PDBChainExtractor

# Same snapshot as protein-only extraction (written after download check)
pdb_data_for_extraction_path = "Results/pdb_data_for_chain_extraction.tsv"
pdb_data = pd.read_csv(pdb_data_for_extraction_path, sep="\t", header=0, engine="python")
pdb_data["Accession"] = pdb_data["Accession"].astype(str).str.upper()
if "Downloaded" in pdb_data.columns:
    pdb_data["Downloaded"] = pdb_data["Downloaded"].astype(str).str.lower().isin(("true", "1"))

chain_extractor = PDBChainExtractor()

# --- Dataset 2: protein chains + nearby small molecules (cofactors, ions, modified residues) ---
# This keeps non-protein atoms within `small_molecule_distance` Å of the chain.
# It also writes `<output>.small_molecules.tsv` inventories alongside each extracted PDB.
chain_extractor.extract_chains_parallel(
    pdb_data,
    target_dir="Results/InterPro_protein_small_molecules/",
    max_workers=None,
    include_small_molecules=True,
    small_molecule_distance=5.0,
    keep_waters=False,
    write_small_molecule_inventory=True,
    show_progress=True,
    show_errors=False,
)

Let's now report on which PDB files contain small molecules.

Let's make sure that the number of structures matches the number of chains in the previous step.

In [ ]:
pdb_directory = 'Results/InterPro_protein_small_molecules/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")

## 2.3 Build KLIFS directories <a id="23"></a>

Two parallel directory sets are prepared here:

1. **InterPro → InterPro+KLIFS merge stubs** — copy the full InterPro-extracted sets into `Results/InterPro_KLIFS_protein_chains/` and `Results/InterPro_KLIFS_protein_small_molecules/`. Downstream steps that operate on the merged InterPro∪KLIFS set can write supplements into these folders later.

2. **Full KLIFS catalogue download** — using `Results/KLIFS/klifs_full_catalog_pdb_chain.tsv` from §1.4 (`build_full_klifs_catalog()`), download every unique KLIFS `(PDB, chain)` from RCSB and write:
   * `Results/KLIFS_protein_chains/` — protein-only chain PDBs (ATOM records)
   * `Results/KLIFS_small_molecules/` — ligand–chain complexes (protein ATOM + qualifying non-water HETATM on that chain)

Full PDB downloads are cached under `Results/KLIFS/rcsb_cache/`. Re-runs skip pairs whose output files already exist; pass `force=True` to `download_all_klifs_chains()` to overwrite.

In [ ]:
import shutil
from workflow.klifs_filter import KLIFSOverlap

# 1) Copy InterPro extracts into InterPro_KLIFS_* merge directories
for src, dst in [
    ("Results/InterPro_protein_chains/",
     "Results/InterPro_KLIFS_protein_chains/"),
    ("Results/InterPro_protein_small_molecules/",
     "Results/InterPro_KLIFS_protein_small_molecules/"),
]:
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        shutil.copy2(os.path.join(src, f), os.path.join(dst, f))

print("Copied InterPro chains and SM files to KLIFS-merged directories.")
print(f"  InterPro_KLIFS_protein_chains/:            "
      f"{count_pdb_files('Results/InterPro_KLIFS_protein_chains/')} PDB files")
print(f"  InterPro_KLIFS_protein_small_molecules/:   "
      f"{count_pdb_files('Results/InterPro_KLIFS_protein_small_molecules/')} PDB files")

# 2) Download all KLIFS (PDB, chain) pairs → dedicated KLIFS-only directories
klifs = KLIFSOverlap()
klifs.build_full_klifs_catalog()  # no-op if already cached from §1.4
klifs.download_all_klifs_chains(
    protein_dir="Results/KLIFS_protein_chains/",
    ligand_dir="Results/KLIFS_small_molecules/",
)

print(f"  KLIFS_protein_chains/:     "
      f"{count_pdb_files('Results/KLIFS_protein_chains/')} PDB files")
print(f"  KLIFS_small_molecules/:    "
      f"{count_pdb_files('Results/KLIFS_small_molecules/')} PDB files")
